# M1: обмен через Google Drive, затем отдельные GPU-проверки

По умолчанию этот notebook проверяет только связь Colab ↔ Mac: устанавливает CPU-пакет, подключает Google Drive и вызывает CLI для preflight и одного настоящего запроса к Docker. CPU runtime достаточно; модель и ML-зависимости не скачиваются.

На Mac должны работать Google Drive, Docker и один `queue-worker` из того же commit. Используйте одинаковый `SESSION_NAME` на обеих машинах: `qwen-repair-exchange/sessions/<SESSION_NAME>`. Команда запуска worker приведена в `docs/m1.md`.

`RUN_INFERENCE` и `RUN_TRAINING` по умолчанию выключены. Включайте соответствующий этап отдельно после проверки связи; для него нужен T4 runtime. Обучение ограничено двумя optimizer step для SFT и GRPO, с сохранением после первого и resume в новом процессе. Эти короткие прогоны не доказывают качество модели.


In [ ]:
REPO_URL = "https://github.com/matveylogee/qwen-repair.git"
COMMIT_SHA = ""  # Full immutable commit, identical on Mac and Colab.
SESSION_NAME = ""  # Same session name used by the Mac worker.
CHECKOUT_DIR = "/content/qwen-repair-m1"
ENV_DIR = "/content/qwen-repair-m1-env"
DRIVE_MOUNT = "/content/drive"
RUN_INFERENCE = False
RUN_TRAINING = False


In [ ]:
import re
import subprocess
import sys
from pathlib import Path
from urllib.parse import urlparse
from uuid import uuid4

transport_ready = False
gpu_ready = False
repo = urlparse(REPO_URL)
if repo.scheme != "https" or not repo.hostname or repo.username or repo.password:
    raise ValueError("Use an HTTPS repository URL without credentials.")
if re.fullmatch(r"[0-9a-fA-F]{40}", COMMIT_SHA) is None:
    raise ValueError("Set the full immutable commit SHA before running.")
if re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", SESSION_NAME) is None:
    raise ValueError("Set SESSION_NAME to the same simple directory name as the Mac worker.")
if type(RUN_INFERENCE) is not bool or type(RUN_TRAINING) is not bool:
    raise ValueError("RUN_INFERENCE and RUN_TRAINING must be booleans.")
checkout = Path(CHECKOUT_DIR)
environment = Path(ENV_DIR)
if checkout.exists() or environment.exists():
    raise FileExistsError("Choose new CHECKOUT_DIR and ENV_DIR to preserve earlier runs.")
subprocess.run(["git", "clone", "--", REPO_URL, str(checkout)], check=True)
subprocess.run(["git", "checkout", "--detach", COMMIT_SHA], cwd=checkout, check=True)
actual_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=checkout, text=True).strip()
if actual_sha.lower() != COMMIT_SHA.lower():
    raise RuntimeError("Checkout does not match the requested commit.")
subprocess.run([sys.executable, "-m", "venv", str(environment)], check=True)
python = str(environment / "bin" / "python")
subprocess.run([python, "-m", "pip", "install", "-e", "."], cwd=checkout, check=True)
run_root = checkout / "runs" / f"m1-{uuid4().hex}"
run_root.mkdir(parents=True)
cli = [python, "-m", "qwen_repair"]


In [ ]:
subprocess.run(cli + ["doctor", "--output", str(run_root / "cpu-environment.json")], cwd=checkout, check=True)


Подключите тот же Google-аккаунт, который используется на Mac. Папка с `SESSION_NAME` появится в Colab после публикации heartbeat worker и синхронизации Drive. Если её нет, проверьте аккаунт, имя сессии и состояние Drive.

Следующая ячейка остановится при неуспешном preflight или probe. Она сохраняет отдельные отчёты и время передачи. После первого запуска с выключенными GPU-этапами скачайте ZIP в последней ячейке и остановите worker на Mac.


In [ ]:
from google.colab import drive

transport_ready = False
drive.mount(DRIVE_MOUNT)
queue_dir = Path(DRIVE_MOUNT) / "MyDrive" / "qwen-repair-exchange" / "sessions" / SESSION_NAME
if not queue_dir.is_dir():
    raise FileNotFoundError("Session folder is missing. Start the Mac worker and wait for Drive synchronization.")
executor_config = run_root / "executor.yaml"
subprocess.run([
    python, "-c",
    "import pathlib,sys,yaml; p=yaml.safe_load(pathlib.Path(sys.argv[1]).read_text()); p['queue_dir']=sys.argv[2]; pathlib.Path(sys.argv[3]).write_text(yaml.safe_dump(p))",
    str(checkout / "configs/colab_queue.yaml"), str(queue_dir), str(executor_config),
], check=True)
subprocess.run(cli + ["executor-preflight", "--config", str(executor_config), "--output", str(run_root / "queue-preflight.json")], cwd=checkout, check=True)
subprocess.run(cli + ["executor-probe", "--config", str(executor_config), "--output", str(run_root / "queue-probe.json")], cwd=checkout, check=True)
transport_ready = True
print((run_root / "queue-probe.json").read_text())


In [ ]:
gpu_ready = False
if RUN_INFERENCE or RUN_TRAINING:
    if not transport_ready:
        raise RuntimeError("Complete the executor preflight and probe before enabling GPU stages.")
    subprocess.run([python, "-m", "pip", "install", "-e", ".[gpu]"], cwd=checkout, check=True)
    subprocess.run(cli + ["doctor", "--gpu", "--output", str(run_root / "gpu-preflight.json")], cwd=checkout, check=True)
    gpu_ready = True
else:
    print("GPU-этапы отключены: проверяем только обмен через Drive.")


In [ ]:
if RUN_INFERENCE:
    if not transport_ready or not gpu_ready:
        raise RuntimeError("Executor and GPU preflight must pass before inference.")
    base_run = run_root / "base"
    subprocess.run(cli + ["infer", "--config", "configs/m1_infer.yaml", "--executor-config", str(executor_config), "--output", str(base_run)], cwd=checkout, check=True)
    subprocess.run(cli + ["summarize", "--run", str(base_run)], cwd=checkout, check=True)


Следующие ячейки выполняются только при `RUN_TRAINING = True`. Для SFT первый процесс делает один шаг при запланированных двух; второй загружает checkpoint и заканчивает тот же план. Восстанавливаются optimizer, scheduler и RNG.


In [ ]:
if RUN_TRAINING:
    if not transport_ready or not gpu_ready:
        raise RuntimeError("Executor and GPU preflight must pass before training.")
    sft_run = run_root / "sft"
    subprocess.run(cli + ["sft-smoke", "--config", "configs/m1_sft.yaml", "--executor-config", str(executor_config), "--output", str(sft_run), "--stop-after", "1"], cwd=checkout, check=True)


In [ ]:
if RUN_TRAINING:
    if not transport_ready or not gpu_ready:
        raise RuntimeError("Executor and GPU preflight must pass before training.")
    subprocess.run(cli + ["sft-smoke", "--config", "configs/m1_sft.yaml", "--executor-config", str(executor_config), "--output", str(sft_run), "--resume-from", str(sft_run / "checkpoint-1")], cwd=checkout, check=True)
    print((sft_run / "training_result.json").read_text())


GRPO: каждый rollout содержит две попытки, в группе два эпизода одной задачи. Feedback исключён из loss, private reward относится только к последнему кандидату. При одинаковых rewards градиент от reward может быть нулевым; это фиксируется в отчёте. Для этой проверки KL отключён явно (`beta=0`).


In [ ]:
if RUN_TRAINING:
    if not transport_ready or not gpu_ready:
        raise RuntimeError("Executor and GPU preflight must pass before training.")
    grpo_config = run_root / "grpo.yaml"
    subprocess.run([
        python, "-c",
        "import pathlib,sys,yaml; p=yaml.safe_load(pathlib.Path(sys.argv[1]).read_text()); p['sft_adapter']=sys.argv[2]; pathlib.Path(sys.argv[3]).write_text(yaml.safe_dump(p))",
        str(checkout / "configs/m1_grpo.yaml"), str(sft_run / "adapter"), str(grpo_config),
    ], check=True)
    grpo_run = run_root / "grpo"
    subprocess.run(cli + ["grpo-smoke", "--config", str(grpo_config), "--executor-config", str(executor_config), "--output", str(grpo_run), "--stop-after", "1"], cwd=checkout, check=True)


In [ ]:
if RUN_TRAINING:
    if not transport_ready or not gpu_ready:
        raise RuntimeError("Executor and GPU preflight must pass before training.")
    subprocess.run(cli + ["grpo-smoke", "--config", str(grpo_config), "--executor-config", str(executor_config), "--output", str(grpo_run), "--resume-from", str(grpo_run / "checkpoint-1")], cwd=checkout, check=True)
    print((grpo_run / "training_result.json").read_text())


Сохраните результаты до завершения runtime. Последняя ячейка упакует каталог запуска в ZIP и начнёт скачивание через Colab. При отключённых GPU-этапах архив содержит только конфигурацию и отчёты проверки связи. При ошибке предыдущей ячейки также выполните упаковку для сохранения доступных отчётов.

Если браузер заблокировал скачивание, используйте панель Files. После проверки остановите `queue-worker` на Mac через Ctrl+C. Checkpoints и результаты создаются на локальном диске Colab; в папку обмена поступают только сообщения executor.


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(str(run_root), "zip", root_dir=run_root.parent, base_dir=run_root.name)
print(f"Archive: {archive}")
files.download(archive)
